# QML 教程：PQC 分类器（unified template）

本教程演示如何使用 `run_pqc_classifier` 训练参数化量子分类器：
1. **Symbolic 编码**：encoding + ansatz 拼成统一模板，**只编译一次**
2. **Autograd 路径**：Simulator + torch 自动微分（快速验证）
3. **Parameter-shift 路径**：Simulator 虚拟后端 + 真机流程（编译 → shot 估计期望值）
4. **IQP 编码**：演示 IQP 编码的非线性特征映射
5. **可视化**：损失收敛、准确率对比

默认用 `Simulator`，便于稳定复现与快速实验。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quantum_hw import QuantumHardwareClient
from quantum_hw.algorithms.qml import run_pqc_classifier
from quantum_hw.api.backend import Backend

def section(title: str):
    print("\n" + "=" * 18, title, "=" * 18)

## 1) 构造训练数据

用 2 个量子比特做二分类。训练数据直接传入 **特征数组**（不再手动构建 QuantumCircuit）：
- 类别 0：特征接近 $[0, 0]$（接近 $|00\rangle$）
- 类别 1：特征接近 $[\pi, \pi]$（接近 $|11\rangle$）

编码方式通过 `encoding` 参数指定（如 `"angle"` / `"iqp"`），内部自动构建 symbolic 线路。

In [ ]:
section("Data preparation")

num_qubits = 2

# (features, label) pairs — 直接传特征数组
train_data = [
    ([0.0, 0.0], 0),
    ([0.2, 0.1], 0),
    ([0.1, 0.3], 0),
    ([np.pi, np.pi], 1),
    ([2.8, 3.0], 1),
    ([3.0, 2.9], 1),
]

print(f"Training samples: {len(train_data)}")
print(f"Num qubits: {num_qubits}")
print(f"Classes: 2 (binary)")
for i, (features, label) in enumerate(train_data):
    print(f"  sample {i}: features={[f'{x:.2f}' for x in features]}  label={label}")

## 2) Autograd 路径（angle encoding）

使用 torch 自动微分计算梯度，速度快、无 shot noise。
`encoding="angle"` 会自动构建 symbolic angle-encoding 线路并与 ansatz 拼接。

In [ ]:
section("Autograd classifier (angle encoding)")

result_autograd = run_pqc_classifier(
    num_qubits=num_qubits,
    train_data=train_data,
    encoding="angle",
    num_classes=2,
    layers=2,
    max_iters=60,
    learning_rate=0.05,
    seed=42,
    gradient_method="autograd",
)

print(f"\nBest loss: {result_autograd.best_loss:.6f}")
print(f"Accuracy:  {result_autograd.accuracy:.2%}")
print(f"Params:    {len(result_autograd.best_params)}")

## 3) Parameter-shift 路径（angle encoding，unified template）

走真机流程：**只编译一次**统一模板 → 每个样本绑定不同特征值 → shot 估计期望值 → parameter-shift 梯度。

与旧版 per-sample 编译相比，编译开销从 $O(N_{\text{samples}})$ 降到 $O(1)$。

**注意**：parameter-shift 每次迭代需执行 $N_{\text{samples}} \times (1 + 2N_{\text{params}})$
次电路，因此比 autograd 慢很多。演示中减少迭代次数。

In [ ]:
section("Parameter-shift classifier (angle encoding, unified template)")

client = QuantumHardwareClient()
backend = Backend("Simulator")

result_ps = run_pqc_classifier(
    num_qubits=num_qubits,
    train_data=train_data,
    encoding="angle",
    num_classes=2,
    layers=2,
    max_iters=30,
    learning_rate=0.05,
    seed=42,
    gradient_method="parameter-shift",
    client=client,
    backend=backend,
    chip_name="Simulator",
    shots=8192,
)

print(f"\nBest loss: {result_ps.best_loss:.6f}")
print(f"Accuracy:  {result_ps.accuracy:.2%}")
print(f"Params:    {len(result_ps.best_params)}")

## 4) IQP 编码：捕捉特征间非线性关系

IQP encoding 在 $H \to R_Z(x_i) \to R_{ZZ}(x_i x_j) \to H$ 结构中引入特征乘积项，
能表达更丰富的特征交互。用法只需改 `encoding="iqp"`。

> **注意**：本演示数据集为简单的线性可分数据（两簇分别位于 $[0,0]$ 和 $[\pi,\pi]$ 附近），
> angle encoding 天然适配这类数据并能达到 100% 准确率。
> IQP encoding 的优势体现在特征间存在非线性交互的数据集上——对当前玩具数据集，
> IQP 准确率通常在 ~83% 左右。

In [ ]:
section("IQP encoding — autograd")

result_iqp = run_pqc_classifier(
    num_qubits=num_qubits,
    train_data=train_data,
    encoding="iqp",
    num_classes=2,
    layers=2,
    max_iters=60,
    learning_rate=0.1,
    seed=42,
    gradient_method="autograd",
)

print(f"\nBest loss: {result_iqp.best_loss:.6f}")
print(f"Accuracy:  {result_iqp.accuracy:.2%}")

## 5) 收敛对比

绘制三种配置的损失曲线：
- **Angle + autograd**（蓝）：精确梯度，无噪声
- **Angle + parameter-shift**（红）：shot-based 估计，带统计涨落
- **IQP + autograd**（绿）：非线性编码

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss convergence
axes[0].plot(result_autograd.loss_history, "o-", ms=3, color="steelblue",
             label=f"angle+autograd (acc={result_autograd.accuracy:.0%})")
axes[0].plot(result_ps.loss_history, "s-", ms=3, color="crimson",
             label=f"angle+param-shift (acc={result_ps.accuracy:.0%})")
axes[0].plot(result_iqp.loss_history, "^-", ms=3, color="seagreen",
             label=f"iqp+autograd (acc={result_iqp.accuracy:.0%})")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss Convergence")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Zoomed in on later iterations
n_skip = min(10, len(result_autograd.loss_history) // 3)
axes[1].plot(range(n_skip, len(result_autograd.loss_history)),
             result_autograd.loss_history[n_skip:],
             "o-", ms=3, color="steelblue", label="angle+autograd")
if len(result_ps.loss_history) > n_skip:
    axes[1].plot(range(n_skip, len(result_ps.loss_history)),
                 result_ps.loss_history[n_skip:],
                 "s-", ms=3, color="crimson", label="angle+param-shift")
axes[1].plot(range(n_skip, len(result_iqp.loss_history)),
             result_iqp.loss_history[n_skip:],
             "^-", ms=3, color="seagreen", label="iqp+autograd")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Loss")
axes[1].set_title("Loss (later iterations)")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Angle+autograd    — final loss: {result_autograd.loss_history[-1]:.6f}  accuracy: {result_autograd.accuracy:.2%}")
print(f"Angle+param-shift — final loss: {result_ps.loss_history[-1]:.6f}  accuracy: {result_ps.accuracy:.2%}")
print(f"IQP+autograd      — final loss: {result_iqp.loss_history[-1]:.6f}  accuracy: {result_iqp.accuracy:.2%}")

## 6) 实验开销分析

parameter-shift 每次迭代的电路执行次数：
$$N_{\text{circuits/iter}} = N_{\text{samples}} \times (1 + 2 \times N_{\text{params}})$$

**Unified template 优势**：编译只做 1 次（而非 $N_{\text{samples}}$ 次），大幅降低真机部署开销。

In [ ]:
section("Cost analysis")

n_samples = len(train_data)
n_params = len(result_ps.best_params)
n_iters_ps = len(result_ps.loss_history)

circuits_per_iter = n_samples * (1 + 2 * n_params)
total_circuits = circuits_per_iter * n_iters_ps

print(f"Samples:          {n_samples}")
print(f"Parameters:       {n_params}")
print(f"Iterations:       {n_iters_ps}")
print(f"Circuits/iter:    {circuits_per_iter}")
print(f"Total circuits:   {total_circuits}")
print(f"Transpilations:   1  (unified template)")

## 下一步建议

| 方向 | 说明 |
|------|------|
| **自定义编码** | 传入 callable `(num_qubits, num_features) -> (circuit, param_names)` |
| **encoding_kwargs** | 如 `encoding_kwargs={"gate": "rx"}` 切换旋转门 |
| **增加数据** | 增大训练集观察泛化能力（过拟合 vs 欠拟合） |
| **多分类** | 设置 `num_classes=3`，使用 softmax 输出 |
| **真实硬件** | 将 `Backend("Simulator")` 替换为真机后端 |
| **层数调节** | 增加 `layers` 观察表达能力和训练难度的权衡 |